# Convert Text-to-Music model to be ONNX compatible

In [ ]:
# install dependencies
!python3 -m pip install --upgrade pip
!pip3 install --pre torch
!pip install transformers
!pip install onnx onnxscript
!pip install samplings
!pip install optimum[onnxruntime]
!pip install torchvision


## Test text-to-music to make sure we like it...

In [2]:
# This code started from an example on the model card on Hugging Face
# https://huggingface.co/sander-wood/text-to-music
import torch
from samplings import top_p_sampling, temperature_sampling
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained('sander-wood/text-to-music')
model = AutoModelForSeq2SeqLM.from_pretrained('sander-wood/text-to-music')
model = model

max_length = 128
top_p = 0.9
temperature = 1.0


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
max_length = 1024

text = "This is a traditional Irish dance music."
input_ids = tokenizer(text,
                      return_tensors='pt',
                      truncation=True,
                      max_length=max_length)['input_ids']

decoder_start_token_id = model.config.decoder_start_token_id
eos_token_id = model.config.eos_token_id

decoder_input_ids = torch.tensor([[decoder_start_token_id]])

for t_idx in range(max_length):
    outputs = model(input_ids=input_ids,
    decoder_input_ids=decoder_input_ids)
    probs = outputs.logits[0][-1]
    probs = torch.nn.Softmax(dim=-1)(probs).detach().numpy()
    sampled_id = temperature_sampling(probs=top_p_sampling(probs,
                                                           top_p=top_p,
                                                           return_probs=True),
                                      temperature=temperature)
    decoder_input_ids = torch.cat((decoder_input_ids, torch.tensor([[sampled_id]])), 1)
    if sampled_id!=eos_token_id:
        continue
    else:
        tune = "X:1\n"
        tune += tokenizer.decode(decoder_input_ids[0], skip_special_tokens=True)
        print(tune)
        break

X:1
L:1/8
Q:1/4=180
M:6/8
K:D
 A |:"D" AdB AFA | DFA"G" B2 A |"D" AdB AGF |"Em" GFG"A7" E2 G |"D" AdB AFA | DFA B2 A |
"G" Bcd"A" ecA |1"D" d3 dcB :|2"D" d3 d2 |: g |"D" fef dfd |"G" gfg"A7" ecA |"D" fef dfd |
"A7" ecA A2 g |"D" fef dfd |"G" gfg"A7" ecA |"G" Bcd"A" ecA |1"D" d3 d2 :|2"D" d3 dcB |]



## Export the model

In [53]:
import torch
from transformers import BartForConditionalGeneration, AutoTokenizer

model_id = "sander-wood/text-to-music"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = BartForConditionalGeneration.from_pretrained(model_id)
model.eval()


# compare against original model

text = "This is a traditional Irish dance music."

input_ids = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=1024
)["input_ids"]

decoder_input_ids = torch.tensor([[model.config.decoder_start_token_id]])

with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        decoder_input_ids=decoder_input_ids,
    )

logits = outputs.logits[0, -1]

top = torch.topk(logits, 20)

print("decoder_start_token_id:", model.config.decoder_start_token_id)
print("input_ids:", input_ids.tolist())
print("TOP TOKEN IDS:", top.indices.tolist())
print("TOP LOGITS:", top.values.tolist())

print(
    "TOP TOKENS:",
    [tokenizer.decode([i]) for i in top.indices.tolist()]
)

decoder_start_token_id: 2
input_ids: [[0, 713, 16, 10, 2065, 3445, 3836, 930, 4, 2]]
TOP TOKEN IDS: [574, 35, 500, 12, 104, 791, 328, 448, 226, 42790, 338, 347, 565, 625, 329, 7083, 462, 8827, 46973, 992]
TOP LOGITS: [21.1384334564209, 9.35181713104248, 6.314741134643555, 6.20809268951416, 6.1715521812438965, 5.148890972137451, 4.8589887619018555, 4.797406196594238, 4.508324146270752, 4.479635238647461, 4.426441192626953, 4.261945724487305, 4.244977951049805, 4.213925361633301, 4.211334228515625, 4.197680950164795, 4.152599811553955, 4.0902910232543945, 3.987856388092041, 3.812375068664551]
TOP TOKENS: ['L', ':', 'R', '-', 'S', 'U', '!', 'M', ' L', ' bc', 'r', 'C', 'T', 'ad', 'z', 'Al', 'l', 'BE', ' dB', ' z']


In [29]:
class TextToMusicBartWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.encoder = model.model.encoder

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=False,
        )

        return outputs[0]

encoder = TextToMusicBartWrapper(model)
encoder.eval()

TextToMusicBartWrapper(
  (encoder): BartEncoder(
    (embed_tokens): BartScaledWordEmbedding(50265, 768, padding_idx=1)
    (embed_positions): BartLearnedPositionalEmbedding(1026, 768)
    (layers): ModuleList(
      (0-5): 6 x BartEncoderLayer(
        (self_attn): BartAttention(
          (k_proj): Linear(in_features=768, out_features=768, bias=True)
          (v_proj): Linear(in_features=768, out_features=768, bias=True)
          (q_proj): Linear(in_features=768, out_features=768, bias=True)
          (out_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (activation_fn): GELUActivation()
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (fc2): Linear(in_features=3072, out_features=768, bias=True)
        (final_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
      )
    )
    (layernorm_embedding): La

In [30]:
inputs = tokenizer(
    "Create a simple bass guitar piece",
    return_tensors="pt",
)

input_ids = inputs["input_ids"]
attention_mask = inputs["attention_mask"]

In [31]:
with torch.no_grad():
    encoder_output = encoder(
        input_ids,
        attention_mask,
    )

print(encoder_output.shape)

torch.Size([1, 8, 768])


In [46]:
torch.onnx.export(
    encoder,
    (
        input_ids,
        attention_mask,
    ),
    "encoder_model.onnx",
    input_names=[
        "input_ids",
        "attention_mask",
    ],
    output_names=[
        "last_hidden_state",
    ],
    dynamic_axes={
        "input_ids": {
            0: "batch",
            1: "encoder_sequence",
        },
        "attention_mask": {
            0: "batch",
            1: "encoder_sequence",
        },
        "last_hidden_state": {
            0: "batch",
            1: "encoder_sequence",
        },
    },
    opset_version=17,
    dynamo=False,
)

/tmp/ipykernel_25543/1186794976.py:1: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


In [49]:
model = onnx.load("encoder_model.onnx")

print("INPUTS:")
for x in model.graph.input:
    print(" ", x.name)

print("OUTPUTS:")
for x in model.graph.output:
    print(" ", x.name)

INPUTS:
  input_ids
  attention_mask
OUTPUTS:
  last_hidden_state


In [47]:
class TextToMusicDecoderWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(
      self,
      input_ids,
      encoder_hidden_states,
      encoder_attention_mask,
    ):
      outputs = self.model(
          encoder_outputs=(encoder_hidden_states,),
          attention_mask=encoder_attention_mask,
          decoder_input_ids=input_ids,
          use_cache=False,
          return_dict=False,
      )

      return outputs[0]

In [34]:
decoder = TextToMusicDecoderWrapper(model)
decoder.eval()

TextToMusicDecoderWrapper(
  (model): BartForConditionalGeneration(
    (model): BartModel(
      (shared): BartScaledWordEmbedding(50265, 768, padding_idx=1)
      (encoder): BartEncoder(
        (embed_tokens): BartScaledWordEmbedding(50265, 768, padding_idx=1)
        (embed_positions): BartLearnedPositionalEmbedding(1026, 768)
        (layers): ModuleList(
          (0-5): 6 x BartEncoderLayer(
            (self_attn): BartAttention(
              (k_proj): Linear(in_features=768, out_features=768, bias=True)
              (v_proj): Linear(in_features=768, out_features=768, bias=True)
              (q_proj): Linear(in_features=768, out_features=768, bias=True)
              (out_proj): Linear(in_features=768, out_features=768, bias=True)
            )
            (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
            (activation_fn): GELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (f

In [35]:
decoder_start_token_id = model.config.decoder_start_token_id

decoder_input_ids = torch.tensor(
    [[decoder_start_token_id]],
    dtype=torch.long,
)

In [36]:
with torch.no_grad():
    logits = decoder(
        decoder_input_ids,
        encoder_output,
        attention_mask,
    )

print(logits.shape)

torch.Size([1, 1, 50265])


In [45]:
torch.onnx.export(
    decoder,
    (
        decoder_input_ids,
        encoder_output,
        attention_mask,
    ),
    "decoder_model_merged.onnx",
    input_names=[
      "input_ids",
      "encoder_hidden_states",
      "encoder_attention_mask",
  ],
    output_names=[
        "logits",
    ],
    dynamic_axes={
        "iput_ids": {
            0: "batch",
            1: "decoder_sequence",
        },
        "encoder_hidden_states": {
            0: "batch",
            1: "encoder_sequence",
        },
        "encoder_attention_mask": {
            0: "batch",
            1: "encoder_sequence",
        },
        "logits": {
            0: "batch",
            1: "decoder_sequence",
        },
    },
    opset_version=17,
    dynamo=False,
)

/tmp/ipykernel_25543/1158632856.py:1: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/usr/local/lib/python3.13/dist-packages/torch/onnx/_internal/torchscript_exporter/utils.py:1511: UserWarning: Provided key iput_ids for dynamic axes is not a valid input/output name
  _validate_dynamic_axes(dynamic_axes, model, input_names, output_names)
/usr/local/lib/python3.13/dist-packages/transformers/models/bart/modeling_bart.py:664: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that th

In [38]:
import onnxruntime as ort
import numpy as np

In [39]:
encoder_session = ort.InferenceSession(
    "bart_encoder.onnx",
    providers=["CPUExecutionProvider"],
)

decoder_session = ort.InferenceSession(
    "bart_decoder.onnx",
    providers=["CPUExecutionProvider"],
)

encoder_result = encoder_session.run(
    ["encoder_hidden_states"],
    {
        "input_ids": input_ids.numpy(),
        "attention_mask": attention_mask.numpy(),
    },
)

encoder_hidden_states = encoder_result[0]

In [40]:
decoder_result = decoder_session.run(
    ["logits"],
    {
        "decoder_input_ids": decoder_input_ids.numpy(),
        "encoder_hidden_states": encoder_hidden_states,
        "encoder_attention_mask": attention_mask.numpy(),
    },
)

logits = decoder_result[0]

print(logits.shape)

(1, 1, 50265)


In [ ]:
generated = tokenizer.decode(
    decoder_ids[0],
    skip_special_tokens=True,
)

print("X:1")
print(generated)

### Debugging model shape/size for hugging face model card..

In [23]:
import onnxruntime as ort

session = ort.InferenceSession(
    "bart_decoder.onnx"
)

print("INPUTS:")
for x in session.get_inputs():
    print(x.name, x.shape, x.type)

print("OUTPUTS:")
for x in session.get_outputs():
    print(x.name, x.shape, x.type)

INPUTS:
decoder_input_ids ['batch', 'decoder_sequence'] tensor(int64)
encoder_hidden_states ['batch', 'encoder_sequence', 768] tensor(float)
encoder_attention_mask ['batch', 'encoder_sequence'] tensor(int64)
OUTPUTS:
logits ['batch', 'decoder_sequence', 50265] tensor(float)


In [44]:
import onnx

model = onnx.load("decoder_model_merged.onnx")

print("INPUTS:")
for x in model.graph.input:
    print(" ", x.name)

print("\nOUTPUTS:")
for x in model.graph.output:
    print(" ", x.name)

INPUTS:
  input_ids
  encoder_hidden_states
  encoder_attention_mask

OUTPUTS:
  logits


In [42]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(
    "sander-wood/text-to-music"
)

generation_config = model.generation_config

In [43]:
generation_config

GenerationConfig {
  "bos_token_id": 0,
  "decoder_start_token_id": 2,
  "early_stopping": true,
  "eos_token_id": 2,
  "forced_bos_token_id": 0,
  "forced_eos_token_id": 2,
  "no_repeat_ngram_size": 3,
  "num_beams": 4,
  "pad_token_id": 1
}

AttributeError: config